# Phase 2 — Pivot Token Detection & Labeling

## What this notebook does
Takes the 93 pivot candidate records from Phase 1 and produces a structured dataset for Phase 3.

For each record it:
1. Finds the **exact token index** of the pivot phrase (not just character position)
2. Assigns a **label**: `correction`, `hallucination_pivot`, or `uncertain`
3. Records the **pivot type**: explicit_correction, restart, or hedging

## Input / Output
- **Input:** `/kaggle/working/pivot_research/pivot_candidates.jsonl` (93 records from Phase 1)
- **Output:** `/kaggle/working/pivot_research/phase2_labeled.jsonl` (structured dataset for Phase 3)

## Run order
Run all cells top to bottom. **No GPU needed** — this is pure text processing.

## Cell 1 — Install dependencies and set up paths

In [3]:
# transformers needed for the tokenizer — same one used in Phase 1
# We must use the EXACT same tokenizer so token indices match
# when Phase 3 slices attention matrices
!pip install -q transformers torch

import json, re, os, random
from transformers import AutoTokenizer

# ── Paths ──────────────────────────────────────────────────────────────────
# Input: the 93 pivot candidates saved by Phase 1
PHASE1_PATH = '/kaggle/input/datasets/gunjalgupta/pivot-candidates/pivot_candidates.jsonl'

# Output: fully labeled dataset ready for Phase 3
PHASE2_PATH = '/kaggle/working/pivot_research/phase2_labeled.jsonl'

os.makedirs('/kaggle/working/pivot_research', exist_ok=True)

# ── Tokenizer ──────────────────────────────────────────────────────────────
# IMPORTANT: must match the model used in Phase 1 generation exactly.
# Different tokenizers split words differently — wrong tokenizer = wrong token indices.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Loading tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded successfully')
print(f'Vocab size: {tokenizer.vocab_size:,}')

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully
Vocab size: 151,643


## Cell 2 — Define pivot detection patterns

Each pattern is paired with a **pivot type** label.
This matters for Phase 3: different pivot types might activate different attention heads.
Tracking the type lets us run that as an ablation study.

Three types:
- `explicit_correction` — model directly says it was wrong
- `restart` — model decides to start a new reasoning approach
- `hedging` — model expresses uncertainty mid-stream

In [5]:
# Each tuple: (regex_pattern, pivot_type)
# Ordered most-specific → least-specific to avoid misclassification
PIVOT_PATTERNS = [
    # --- Explicit corrections: model directly acknowledges an error ---
    (r"\bwait,\s",                                           "explicit_correction"),
    (r"\bwait\u2014",                                        "explicit_correction"),  # wait—
    (r"\bactually,\s+i\b",                                   "explicit_correction"),  # Actually, I...
    (r"\bactually,\s+no\b",                                  "explicit_correction"),  # Actually, no
    (r"\bactually,\s+that\b",                                "explicit_correction"),  # Actually, that's wrong
    (r"\bactually,\s+(?:no|let|i|that|this|the|we|my|on)",  "explicit_correction"),  # Actually, [word]
    (r"\bi made an error",                                   "explicit_correction"),
    (r"\bi was wrong",                                       "explicit_correction"),
    (r"\bthat('s| is) (wrong|incorrect|not right)",          "explicit_correction"),
    (r"\bi need to correct",                                 "explicit_correction"),
    (r"\bno,\s+wait",                                        "explicit_correction"),
    (r"\bi('ve| have) made a mistake",                       "explicit_correction"),

    # --- Restarts: model abandons current approach and starts fresh ---
    (r"\blet me reconsider",  "restart"),
    (r"\blet me recalculate", "restart"),
    (r"\blet me re-?read",    "restart"),

    # --- Hedging: model expresses uncertainty mid-stream ---
    (r"\bon second thought",  "hedging"),
    (r"\bhold on",            "hedging"),
    (r"\bhmm,\s",             "hedging"),
]

# Compile into one master regex.
# Each pattern is in a capturing group so we can identify which one fired.
PIVOT_RE = re.compile(
    "|".join(f"({p})" for p, _ in PIVOT_PATTERNS),
    re.IGNORECASE
)

print(f'Loaded {len(PIVOT_PATTERNS)} pivot patterns across 3 types')

# Quick test
test_cases = [
    "I think it's 64. Wait, actually that's not right.",
    "The answer is 45 mph. Hmm, let me reconsider.",
    "So x = 10. Hold on, I made an error.",
]
print('\nRegex test:')
for tc in test_cases:
    m = PIVOT_RE.search(tc)
    print(f'  "{tc[:55]}" -> {repr(m.group()) if m else "NO MATCH"}')

Loaded 18 pivot patterns across 3 types

Regex test:
  "I think it's 64. Wait, actually that's not right." -> 'Wait, '
  "The answer is 45 mph. Hmm, let me reconsider." -> 'Hmm, '
  "So x = 10. Hold on, I made an error." -> 'Hold on'


## Cell 3 — Token-level pivot finder

This is the most critical function in Phase 2.

**Why token indices, not character indices?**
Phase 3 extracts attention matrices using TransformerLens.
Those matrices are indexed by token position, not character position.
If we record the wrong index, Phase 3 slices the attention matrix at the
wrong position and the entire analysis is misaligned.

**The tokenization trap:**
`Actually` might tokenize as one token (`▁Actually`) or two (`▁Act` + `ually`).
We must map character positions to token positions carefully.

In [6]:
def find_pivot_token_idx(text: str) -> list:
    """
    Find all pivot phrases in text and return their exact token positions.

    Returns list of dicts, one per detected pivot:
      char_start        character index where pivot phrase starts
      char_end          character index where pivot phrase ends
      phrase            the matched text e.g. 'Wait, '
      pivot_type        explicit_correction | restart | hedging
      token_start       token index of FIRST token of pivot phrase
      token_end         token index of LAST token of pivot phrase
      pre_pivot_tokens  number of tokens BEFORE the pivot
                        (Phase 3 analyzes these for attention patterns)
      total_tokens      total token count of the full text
    """
    # Step 1: tokenize the full generated text
    tokens     = tokenizer.encode(text, add_special_tokens=False)
    token_strs = [tokenizer.decode([t]) for t in tokens]

    # Step 2: build character_index -> token_index map
    # Replay token strings to track which chars each token covers
    char_to_token = {}
    cursor = 0
    for i, tok in enumerate(token_strs):
        for j in range(len(tok)):
            if cursor + j not in char_to_token:   # first token wins
                char_to_token[cursor + j] = i
        cursor += len(tok)

    # Step 3: find all regex matches
    detections = []
    for match in PIVOT_RE.finditer(text):

        # Find which capturing group fired to get the pivot type
        matched_type = "unknown"
        for group_idx, (pattern, pivot_type) in enumerate(PIVOT_PATTERNS):
            if match.group(group_idx + 1) is not None:
                matched_type = pivot_type
                break

        char_start = match.start()
        char_end   = match.end() - 1

        # Step 4: map char positions to token positions
        token_start = char_to_token.get(char_start)
        token_end   = char_to_token.get(char_end)

        # Skip if mapping failed (rare edge case with unusual whitespace)
        if token_start is None or token_end is None:
            continue

        # Skip pivots too close to start — Phase 3 needs at least 15 pre-pivot tokens
        if token_start < 15:
            continue

        detections.append({
            'char_start':       char_start,
            'char_end':         char_end,
            'phrase':           match.group(),
            'pivot_type':       matched_type,
            'token_start':      token_start,
            'token_end':        token_end,
            'pre_pivot_tokens': token_start,
            'total_tokens':     len(tokens),
        })

    return detections


# ── Test on first Phase 1 record ───────────────────────────────────────────
with open(PHASE1_PATH, 'r') as f:
    sample = json.loads(f.readline())

detections = find_pivot_token_idx(sample['generated_text'])
print(f'Prompt: {sample["prompt_id"]}')
print(f'Found {len(detections)} pivot(s):')
for d in detections:
    print(f"  phrase:           '{d['phrase'].strip()}'")
    print(f"  type:             {d['pivot_type']}")
    print(f"  token_start:      {d['token_start']}")
    print(f"  pre_pivot_tokens: {d['pre_pivot_tokens']}")
    print(f"  total_tokens:     {d['total_tokens']}")
    print()

Prompt: mis_01
Found 1 pivot(s):
  phrase:           'Actually, that'
  type:             explicit_correction
  token_start:      25
  pre_pivot_tokens: 25
  total_tokens:     118



## Cell 4 — Automatic labeler

Labels each detected pivot as one of:
- `correction` — wrong before pivot, right after
- `hallucination_pivot` — model drifted away from the correct answer
- `uncertain` — correct answer appears both sides (stylistic), needs human review

**Method:** keyword matching against ground truth answers.

**Limitation:** automatic labels are ~80% accurate.
For the final paper, uncertain cases + a random 20% sample should be
human-reviewed to compute inter-annotator agreement (Cohen's Kappa > 0.7).

In [7]:
# Ground truth keyword map: prompt_id -> keywords indicating a correct answer
# Any keyword appearing in post-pivot text = model reached the right answer
GROUND_TRUTH = {
    # Misdirection
    "mis_01": ["roosters don't lay", "rooster", "can't lay", "cannot lay", "doesn't lay"],
    "mis_02": ["noah", "not columbus", "columbus didn't"],
    "mis_03": ["don't bury", "survivors", "can't bury survivors", "you don't bury"],
    "mis_04": ["one hour", "60 minutes", "1 hour"],
    "mis_05": ["second", "2nd", "last place"],
    "mis_06": ["you have 4", "you took 4", "4 oranges"],
    "mis_07": ["7", "seven cows"],
    "mis_08": ["match", "light the match", "match first"],
    "mis_09": ["5 cents", "$0.05", "0.05"],
    "mis_10": ["all", "every", "12", "all of them"],
    # Math
    "math_01": ["40 mph", "40mph", "harmonic"],
    "math_02": ["8 days", "eight days"],
    "math_03": ["44%", "44 percent", "not 50"],
    "math_04": ["75", "$75", "not 100"],
    "math_05": ["6 days", "six days"],
    "math_06": ["499", "499 bricks"],
    "math_07": ["80 gallons", "80-gallon"],
    "math_08": ["8 days", "eight days"],
    "math_09": ["120 miles", "120 mile"],
    "math_10": ["50%", "50 percent", "still 50", "independent", "no effect", 
            "gambler's fallacy", "does not change", "same probability"],
    "math_11": ["7.5", "7.5 degrees"],
    "math_12": ["21", "twenty-one"],
    "math_13": ["25 feet", "25-foot"],
    "math_14": ["7 minutes", "seven minutes"],
    "math_15": ["128", "128 layers"],
    "math_16": ["8 feet", "8-foot"],
    "math_17": ["1/6", "16.7%", "six ways"],
    "math_18": ["30", "thirty"],
    "math_19": ["204", "204 squares"],
    "math_20": ["90", "ninety"],
    "math_21": ["friday", "Friday"],
    "math_22": ["3.5", "three point five"],
    "math_23": ["62.5%", "5/8"],
    "math_24": ["15", "fifteen handshakes"],
    "math_25": ["no", "not necessarily", "12 is"],
    "math_26": ["impossible", "can't", "cannot", "never"],
    "math_27": ["2/15", "13.3%"],
    "math_28": ["24", "twenty-four"],
    "math_29": ["no", "not a right", "not right triangle"],
    "math_30": ["1331", "$1331"],
    "math_31": ["golden ratio", "1.618", "phi"],
    "math_32": ["488", "not 600"],
    "math_33": ["about 16", "0.159"],
    "math_34": ["24", "twenty-four zeros"],
    "math_35": ["301", "three hundred"],
    "math_36": ["good bet", "50%", "50.7%", "probable"],
    "math_37": ["one piece", "1 piece", "still connected"],
    "math_38": ["equal", "equals 1", "same as 1"],
    "math_39": ["5 cents", "$0.05", "0.05"],
    "math_40": ["50%", "50.7%", "about 50"],
    # Logic
    "logic_01": ["switch", "yes", "should switch", "2/3"],
    "logic_02": ["probably not", "less than 1%", "unlikely", "false positive"],
    "logic_03": ["librarian", "representativeness", "base rate"],
    "logic_04": ["50 cents and 5 cents", "half dollar", "50-cent"],
    "logic_05": ["10", "son is 10"],
    "logic_06": ["yes", "can be", "is a"],
    "logic_07": ["mixed box", "both", "apple-orange"],
    "logic_08": ["1/3", "33%", "one third"],
    "logic_09": ["fish bowl", "aquarium", "fishbowl"],
    "logic_10": ["3 weighings", "possible", "can be done"],
    "logic_11": ["yes", "must be", "there is always"],
    "logic_12": ["turn on", "leave one on", "heat"],
    "logic_13": ["no", "flawed", "not valid"],
    "logic_14": ["yes", "can guarantee", "strategy"],
    "logic_15": ["what would the other guard say", "same door", "other guard"],
    "logic_16": ["2/3", "two thirds", "0.667"],
    "logic_17": ["german", "fifth house"],
    "logic_18": ["40", "forty"],
    "logic_19": ["possible", "can be done", "not impossible"],
    "logic_20": ["photographer", "photo", "picture"],
}


def auto_label(record: dict, pivot: dict) -> dict:
    """
    Label a pivot as correction, hallucination_pivot, or uncertain.

    Logic:
      Split generated text at pivot character position.
      Check whether correct-answer keywords appear before vs after pivot.

      wrong before + right after  ->  correction
      right before + wrong after  ->  hallucination_pivot (drifted away from answer)
      wrong both sides            ->  hallucination_pivot (pivot changed nothing)
      right both sides            ->  uncertain (stylistic, not a real correction)
    """
    keywords = GROUND_TRUTH.get(record['prompt_id'], [])
    if not keywords:
        return {"label": "uncertain", "reason": "no ground truth for this prompt"}

    # Split at the pivot character position
    gen       = record['generated_text']
    pre_text  = gen[:pivot['char_start']].lower()
    post_text = gen[pivot['char_start']:].lower()

    pre_correct  = any(kw.lower() in pre_text  for kw in keywords)
    post_correct = any(kw.lower() in post_text for kw in keywords)

    if   not pre_correct and post_correct:
        return {"label": "correction",
                "reason": "wrong before pivot, correct answer appears after"}
    elif pre_correct and not post_correct:
        return {"label": "hallucination_pivot",
                "reason": "correct answer present before pivot but lost after"}
    elif not pre_correct and not post_correct:
        return {"label": "hallucination_pivot",
                "reason": "wrong both before and after pivot"}
    else:
        return {"label": "uncertain",
                "reason": "correct answer present both sides — likely stylistic pivot"}


# ── Test on first record ───────────────────────────────────────────────────
with open(PHASE1_PATH, 'r') as f:
    sample = json.loads(f.readline())

pivots = find_pivot_token_idx(sample['generated_text'])
if pivots:
    result = auto_label(sample, pivots[0])
    print(f"Prompt:  {sample['prompt_id']}")
    print(f"Phrase:  '{pivots[0]['phrase'].strip()}'")
    print(f"Label:   {result['label']}")
    print(f"Reason:  {result['reason']}")
else:
    print('No pivots found in sample — check PHASE1_PATH is correct')

Prompt:  mis_01
Phrase:  'Actually, that'
Label:   hallucination_pivot
Reason:  wrong both before and after pivot


## Cell 5 — Run Phase 2 on all 93 records

Processes every record and writes the labeled dataset.

For each record:
- Run token-level pivot detection
- Take only the **first** pivot (most pre-pivot context for Phase 3)
- Run automatic labeler
- Write full structured record to `phase2_labeled.jsonl`

In [7]:
# Load all Phase 1 records
with open(PHASE1_PATH, 'r') as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f'Loaded {len(records)} records from Phase 1')
print('Processing...')

results      = []
skipped      = []
label_counts = {"correction": 0, "hallucination_pivot": 0, "uncertain": 0}

for record in records:

    # Find pivot token indices
    pivots = find_pivot_token_idx(record['generated_text'])

    if not pivots:
        # Token-level detection found nothing — skip
        skipped.append(record['prompt_id'])
        continue

    # Use FIRST pivot only — earliest reasoning shift,
    # gives Phase 3 the most pre-pivot context to analyze
    pivot = pivots[0]
    label = auto_label(record, pivot)

    result = {
        # From Phase 1
        "prompt_id":         record['prompt_id'],
        "category":          record['category'],
        "prompt_text":       record['prompt_text'],
        "sample_idx":        record['sample_idx'],
        "generated_text":    record['generated_text'],
        "model_name":        record['model_name'],
        "seed":              record['seed'],
        "timestamp":         record['timestamp'],
        # Phase 2: pivot location (used by Phase 3 to slice attention matrices)
        "pivot_phrase":      pivot['phrase'],
        "pivot_type":        pivot['pivot_type'],
        "pivot_token_start": pivot['token_start'],
        "pivot_token_end":   pivot['token_end'],
        "pre_pivot_tokens":  pivot['pre_pivot_tokens'],
        "total_tokens":      pivot['total_tokens'],
        # Phase 2: label
        "label":             label['label'],
        "label_reason":      label['reason'],
        "label_method":      "automatic",
        "human_verified":    False,   # set True after manual review
    }

    results.append(result)
    label_counts[label['label']] += 1

# Save to disk
with open(PHASE2_PATH, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')

# Summary
print(f'\nInput:   {len(records)} records')
print(f'Output:  {len(results)} records')
print(f'Skipped: {len(skipped)}  (no token-level pivot found)')
if skipped:
    print(f'  IDs: {skipped[:10]}')
print()
print('Label distribution:')
for lbl, count in label_counts.items():
    pct = count / len(results) * 100 if results else 0
    bar = chr(9608) * int(pct / 3)
    print(f'  {lbl:<22} {count:>3}  ({pct:5.1f}%)  {bar}')
print(f'\nSaved to: {PHASE2_PATH}')

# Warn if either class is too small for Phase 4 classifier
if label_counts['correction'] < 20:
    print('WARNING: fewer than 20 correction labels — consider collecting more Phase 1 data')
if label_counts['hallucination_pivot'] < 20:
    print('WARNING: fewer than 20 hallucination_pivot labels — consider collecting more data')

Loaded 93 records from Phase 1
Processing...

Input:   93 records
Output:  93 records
Skipped: 0  (no token-level pivot found)

Label distribution:
  correction               8  (  8.6%)  ██
  hallucination_pivot     59  ( 63.4%)  █████████████████████
  uncertain               26  ( 28.0%)  █████████

Saved to: /kaggle/working/pivot_research/phase2_labeled.jsonl


# Override the uncertains manually

In [8]:
# Fill this in after reading the uncertain samples above
# Format: "prompt_id#sample_idx" -> "correction" or "hallucination_pivot"
# Leave out any you genuinely can't decide

MANUAL_OVERRIDES = {
    # mis_01 #10 — model correctly says roosters don't lay eggs, pivot is stylistic restatement
    "mis_01#10":  "uncertain",       # leave out — genuinely ambiguous

    # mis_04 #13 — had 60 min (correct), pivot reconsiders → still correct. Stylistic.
    "mis_04#13":  "uncertain",       # leave out

    # mis_06 #11 — says "10 oranges left" (WRONG — answer is 4), pivot doesn't fix it
    "mis_06#11":  "hallucination_pivot",

    # mis_06 #13 — pivot correctly lands on "4 oranges" after confusion
    "mis_06#13":  "correction",

    # mis_06 #14 — had correct answer (6... wait, the answer is 4 you took away)
    # "you have 4" because you TOOK 4. Pivot doesn't fix it.
    "mis_06#14":  "hallucination_pivot",

    # mis_07 #7 — starts with wrong answer (13), pivot corrects to 7 ✓
    "mis_07#7":   "correction",

    # mis_07 #14 — starts with wrong answer (13), pivot corrects to 7 ✓
    "mis_07#14":  "correction",

    # mis_10 #9 — starts counting only Feb, pivot opens up to "all months have 28 days"
    "mis_10#9":   "correction",

    # mis_10 #11 — starts with "zero months", pivot corrects to "all 12" ✓
    "mis_10#11":  "correction",

    # mis_10 #12 — starts with "only February", pivot questions it but stays wrong
    "mis_10#12":  "hallucination_pivot",

    # mis_10 #14 — starts with "none", pivot corrects to "February has 28 days" — partial correct
    "mis_10#14":  "correction",

    # math_01 #9 — already had 40 mph (correct), pivot is just restating. Stylistic.
    "math_01#9":  "uncertain",       # leave out

    # math_11 #8 — had 18.75° (correct answer is 7.5°), pivot recalculates but unclear
    "math_11#8":  "hallucination_pivot",

    # math_16 #13 — had 8 feet (CORRECT), pivot says "too low" and doubts right answer
    "math_16#13": "hallucination_pivot",

    # math_18 #13 — had 30 (correct), pivot doubts it unnecessarily
    "math_18#13": "hallucination_pivot",

    # math_23 #2 — had 62.5% (correct), pivot says "doesn't seem right" and drifts
    "math_23#2":  "hallucination_pivot",

    # math_25 #5 — pivot is just the system-prompt phrase, not real correction
    "math_25#5":  "uncertain",       # leave out

    # math_29 #8 — had correct answer (not right triangle), pivot doubts it unnecessarily
    "math_29#8":  "hallucination_pivot",

    # math_29 #9 — had correct answer, pivot claims arithmetic error but there isn't one
    "math_29#9":  "hallucination_pivot",

    # math_30 #6 — had $1331 (correct), pivot doubts it then re-confirms. Stylistic.
    "math_30#6":  "uncertain",       # leave out

    # math_38 #6 — had correct answer (=1), pivot says "doesn't feel right" and drifts
    "math_38#6":  "hallucination_pivot",

    # math_38 #12 — had correct answer (=1), pivot confirms it. Stylistic restatement.
    "math_38#12": "uncertain",       # leave out

    # logic_01 #7 — had partial correct reasoning, pivot introduces doubt without fixing
    "logic_01#7": "hallucination_pivot",

    # logic_03 #10 — had "librarian" (correct), pivot drifts to teacher/architect
    "logic_03#10": "hallucination_pivot",

    # logic_12 #5 — pivot is literally the system prompt text echoed back, not real
    "logic_12#5": "uncertain",       # leave out

    # logic_15 #9 — had correct answer, pivot says "not quite correct" and drifts
    "logic_15#9": "hallucination_pivot",
}

# Apply overrides and resave
for r in results:
    key = f"{r['prompt_id']}#{r['sample_idx']}"
    if key in MANUAL_OVERRIDES:
        r['label']          = MANUAL_OVERRIDES[key]
        r['label_method']   = 'human'
        r['human_verified'] = True

# Recount
from collections import Counter
new_counts = Counter(r['label'] for r in results)
print('Updated label distribution:')
for lbl, count in new_counts.items():
    print(f'  {lbl:<22} {count}')

# Resave with overrides applied
with open(PHASE2_PATH, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print(f'\nResaved to {PHASE2_PATH}')

Updated label distribution:
  hallucination_pivot    72
  uncertain              7
  correction             14

Resaved to /kaggle/working/pivot_research/phase2_labeled.jsonl


## Cell 6 — Sanity check: read samples from each label class

Print 3 random records per label so you can manually verify the labels look correct.
Read the context around each pivot phrase — does the label match what you see?

This is important: automatic labels are ~80% accurate.
Reviewing these samples gives you a gut check before Phase 3.

In [9]:
random.seed(42)   # fixed seed — same samples every run for reproducibility

for label in ["correction", "hallucination_pivot", "uncertain"]:
    subset = [r for r in results if r['label'] == label]

    print(f"{'='*65}")
    print(f"  {label.upper()}  ({len(subset)} total)")
    print(f"{'='*65}")

    if not subset:
        print('  (none)\n')
        continue

    for r in random.sample(subset, min(3, len(subset))):
        print(f"\n  [{r['prompt_id']} #{r['sample_idx']}] — {r['pivot_type']}")
        print(f"  Prompt:     {r['prompt_text'][:75]}")
        print(f"  Pivot:      '{r['pivot_phrase'].strip()}' @ token {r['pivot_token_start']}")
        print(f"  Pre-tokens: {r['pre_pivot_tokens']}")
        print(f"  Reason:     {r['label_reason']}")

        # Show 100 chars before and 150 chars after the pivot for context
        gen         = r['generated_text']
        pivot_char  = gen.lower().find(r['pivot_phrase'].lower())
        if pivot_char >= 0:
            before = gen[max(0, pivot_char - 100):pivot_char]
            phrase = gen[pivot_char:pivot_char + len(r['pivot_phrase'])]
            after  = gen[pivot_char + len(r['pivot_phrase']):pivot_char + len(r['pivot_phrase']) + 150]
            print(f"  Context:    ...{before}[[ {phrase} ]]{after}...")
    print()

  CORRECTION  (14 total)

  [logic_01 #2] — explicit_correction
  Prompt:     You're on a game show. Three doors: one hides a car, two hide goats. You pi
  Pivot:      'Wait,' @ token 276
  Pre-tokens: 276
  Reason:     wrong before pivot, correct answer appears after
  Context:    ...ing in door 1 or door 3, assuming we stick with our initial choice.

3. **Pause Mid-way and Ask:** "[[ Wait,  ]]is this actually correct?" 

   Yes, that analysis seems sound at first glance. However, let's consider whether switching might provide a better strat...

  [mis_06 #13] — explicit_correction
  Prompt:     There are 10 oranges in a bowl. You take away 4. How many oranges do you ha
  Pivot:      'wait,' @ token 64
  Pre-tokens: 64
  Reason:     correct answer present both sides — likely stylistic pivot
  Context:    ... there might be an error lurking somewhere, especially given the nature of the numbers involved. Oh [[ wait,  ]]I realized! We started with a full bag of 10 oranges, and we've take

## Cell 7 — Pivot type and category breakdown

Shows how pivot types distribute across label classes.
If `explicit_correction` pivots produce more genuine corrections than `hedging` pivots,
that itself is a reportable finding for the paper.

In [10]:
print('Pivot type x label breakdown:')
print(f"{'pivot_type':<22} {'correction':>12} {'halluc_pivot':>14} {'uncertain':>11} {'total':>7}")
print('-' * 68)

for pt in sorted(set(r['pivot_type'] for r in results)):
    sub    = [r for r in results if r['pivot_type'] == pt]
    corr   = sum(1 for r in sub if r['label'] == 'correction')
    halluc = sum(1 for r in sub if r['label'] == 'hallucination_pivot')
    uncert = sum(1 for r in sub if r['label'] == 'uncertain')
    print(f"  {pt:<20} {corr:>12} {halluc:>14} {uncert:>11} {len(sub):>7}")

print('-' * 68)
print(f"  {'TOTAL':<20} "
      f"{label_counts['correction']:>12} "
      f"{label_counts['hallucination_pivot']:>14} "
      f"{label_counts['uncertain']:>11} "
      f"{len(results):>7}")

print()
print('Category x label breakdown:')
print(f"{'category':<16} {'correction':>12} {'halluc_pivot':>14} {'uncertain':>11} {'total':>7}")
print('-' * 62)
for cat in ['misdirection', 'math', 'logic']:
    sub    = [r for r in results if r['category'] == cat]
    corr   = sum(1 for r in sub if r['label'] == 'correction')
    halluc = sum(1 for r in sub if r['label'] == 'hallucination_pivot')
    uncert = sum(1 for r in sub if r['label'] == 'uncertain')
    print(f"  {cat:<14} {corr:>12} {halluc:>14} {uncert:>11} {len(sub):>7}")

Pivot type x label breakdown:
pivot_type               correction   halluc_pivot   uncertain   total
--------------------------------------------------------------------
  explicit_correction            13             68           6      87
  hedging                         0              1           1       2
  unknown                         1              3           0       4
--------------------------------------------------------------------
  TOTAL                           8             59          26      93

Category x label breakdown:
category           correction   halluc_pivot   uncertain   total
--------------------------------------------------------------
  misdirection              7             12           2      21
  math                      3             38           4      45
  logic                     4             22           1      27


## Cell 8 — Confirm output and download

Confirms the output file is saved and ready to download.

Download `phase2_labeled.jsonl` from **Kaggle sidebar → Output tab**.
This is the input file for Phase 3 (attention head extraction).

In [11]:
import json
from collections import Counter

with open('/kaggle/working/pivot_research/phase2_labeled.jsonl') as f:
    records = [json.loads(line) for line in f]

print(f'Total records: {len(records)}')
print('Label distribution:')
for lbl, count in Counter(r['label'] for r in records).items():
    print(f'  {lbl:<22} {count}')

usable = [r for r in records if r['label'] in ['correction', 'hallucination_pivot']]
print(f'\nUsable for Phase 3/4: {len(usable)}')

Total records: 93
Label distribution:
  hallucination_pivot    72
  uncertain              7
  correction             14

Usable for Phase 3/4: 86


In [12]:
size_mb    = os.path.getsize(PHASE2_PATH) / 1e6
line_count = sum(1 for _ in open(PHASE2_PATH))

print('Phase 2 complete!')
print(f'  File:    {PHASE2_PATH}')
print(f'  Size:    {size_mb:.2f} MB')
print(f'  Records: {line_count}')
print()
print('Fields in each output record:')
with open(PHASE2_PATH) as f:
    s = json.loads(f.readline())
for k, v in s.items():
    print(f'  {k:<22}  {str(v)[:55]}')
print()
print('To download: Kaggle sidebar -> Output tab -> phase2_labeled.jsonl -> Download')
print('This file is the input for Phase 3 (attention head extraction).')

Phase 2 complete!
  File:    /kaggle/working/pivot_research/phase2_labeled.jsonl
  Size:    0.19 MB
  Records: 93

Fields in each output record:
  prompt_id               mis_01
  category                misdirection
  prompt_text             A rooster sits on a barn roof. It lays an egg. The wind
  sample_idx              5
  generated_text          A 10% increase on $100 gives $110. Then a 10% decrease.
  model_name              Qwen/Qwen2.5-1.5B-Instruct
  seed                    232654075
  timestamp               2026-05-22T23:51:32.917845
  pivot_phrase            Actually, that
  pivot_type              explicit_correction
  pivot_token_start       25
  pivot_token_end         27
  pre_pivot_tokens        25
  total_tokens            118
  label                   hallucination_pivot
  label_reason            wrong both before and after pivot
  label_method            automatic
  human_verified          False

To download: Kaggle sidebar -> Output tab -> phase2_labeled.jsonl -> D

In [14]:
uncertain = [r for r in results if r['label'] == 'uncertain']

print(f'Manually review these {len(uncertain)} uncertain records.')
print('For each one, decide: is the pivot a genuine correction or hallucination?\n')

for i, r in enumerate(uncertain):
    print(f"[{i+1}/{len(uncertain)}] {r['prompt_id']} #{r['sample_idx']}")
    print(f"  Prompt: {r['prompt_text']}")
    print(f"  Pivot:  '{r['pivot_phrase'].strip()}' ({r['pivot_type']})")
    gen        = r['generated_text']
    pivot_char = gen.lower().find(r['pivot_phrase'].lower())
    if pivot_char >= 0:
        before = gen[max(0, pivot_char-150):pivot_char]
        after  = gen[pivot_char:pivot_char+200]
        print(f"  Before: ...{before}")
        print(f"  After:  {after}...")
    print()

Manually review these 26 uncertain records.
For each one, decide: is the pivot a genuine correction or hallucination?

[1/26] mis_01 #10
  Prompt: A rooster sits on a barn roof. It lays an egg. The wind is blowing east. Which direction does the egg roll?
  Pivot:  'Wait,' (explicit_correction)
  Before: ...ster, it will naturally fall towards the earth under the influence of gravity.

Therefore, the egg will roll down towards the earth.

Explicit pause: 
  After:  Wait, actually, there seems to be a misunderstanding here. Remembering the principles of physics, especially that gravity causes things to fall downward rather than moving horizontally unless propelle...

[2/26] mis_04 #13
  Prompt: A doctor gives you 4 pills. Take one every 20 minutes. How long until they are all gone?
  Pivot:  'Let me reconsider' (hedging)
  Before: ... 60 \text{ minutes} \]

So, it would take 60 minutes to finish taking all 4 pills under these conditions.

Double-checking with explicit doubt: Wait! 
  Aft

In [3]:
import json, re, os, random
from transformers import AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded')

PIVOT_PATTERNS = [
    (r"\bwait,\s", "explicit_correction"),
    (r"\bwait\u2014", "explicit_correction"),
    (r"\bactually,\s+i\b", "explicit_correction"),
    (r"\bactually,\s+no\b", "explicit_correction"),
    (r"\bactually,\s+that\b", "explicit_correction"),
    (r"\bactually,\s+(?:no|let|i|that|this|the|we|my|on)", "explicit_correction"),
    (r"\bi made an error", "explicit_correction"),
    (r"\bi was wrong", "explicit_correction"),
    (r"\bthat('s| is) (wrong|incorrect|not right)", "explicit_correction"),
    (r"\bi need to correct", "explicit_correction"),
    (r"\bno,\s+wait", "explicit_correction"),
    (r"\bi('ve| have) made a mistake", "explicit_correction"),
    (r"\blet me reconsider", "restart"),
    (r"\blet me recalculate", "restart"),
    (r"\blet me re-?read", "restart"),
    (r"\bon second thought", "hedging"),
    (r"\bhold on", "hedging"),
    (r"\bhmm,\s", "hedging"),
]
PIVOT_RE = re.compile("|".join(f"({p})" for p, _ in PIVOT_PATTERNS), re.IGNORECASE)

def find_pivot_token_idx(text: str) -> list:
    tokens     = tokenizer.encode(text, add_special_tokens=False)
    token_strs = [tokenizer.decode([t]) for t in tokens]
    char_to_token = {}
    cursor = 0
    for i, tok in enumerate(token_strs):
        for j in range(len(tok)):
            if cursor + j not in char_to_token:
                char_to_token[cursor + j] = i
        cursor += len(tok)

    detections = []
    for match in PIVOT_RE.finditer(text):
        matched_type = "unknown"
        for group_idx, (pattern, pivot_type) in enumerate(PIVOT_PATTERNS):
            if match.group(group_idx + 1) is not None:
                matched_type = pivot_type
                break
        char_start = match.start()
        char_end   = match.end() - 1
        token_start = char_to_token.get(char_start)
        token_end   = char_to_token.get(char_end)
        if token_start is None or token_end is None:
            continue
        if token_start < 15:
            continue
        detections.append({
            'char_start': char_start, 'char_end': char_end,
            'phrase': match.group(), 'pivot_type': matched_type,
            'token_start': token_start, 'token_end': token_end,
            'pre_pivot_tokens': token_start, 'total_tokens': len(tokens),
        })
    return detections

print('find_pivot_token_idx defined')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded
find_pivot_token_idx defined


In [4]:
GROUND_TRUTH = {
    "mis_01": ["roosters don't lay", "rooster", "can't lay", "cannot lay", "doesn't lay"],
    "mis_02": ["noah", "not columbus", "columbus didn't"],
    "mis_03": ["don't bury", "survivors", "can't bury survivors", "you don't bury"],
    "mis_04": ["one hour", "60 minutes", "1 hour"],
    "mis_05": ["second", "2nd", "last place"],
    "mis_06": ["you have 4", "you took 4", "4 oranges"],
    "mis_07": ["7", "seven cows"],
    "mis_08": ["match", "light the match", "match first"],
    "mis_09": ["5 cents", "$0.05", "0.05"],
    "mis_10": ["all", "every", "12", "all of them"],
    "math_01": ["40 mph", "40mph", "harmonic"],
    "math_02": ["8 days", "eight days"],
    "math_03": ["44%", "44 percent", "not 50"],
    "math_04": ["75", "$75", "not 100"],
    "math_05": ["6 days", "six days"],
    "math_06": ["499", "499 bricks"],
    "math_07": ["80 gallons", "80-gallon"],
    "math_08": ["8 days", "eight days"],
    "math_09": ["120 miles", "120 mile"],
    "math_10": ["50%", "50 percent", "still 50", "independent", "no effect",
                "gambler's fallacy", "does not change", "same probability"],
    "math_11": ["7.5", "7.5 degrees"],
    "math_12": ["21", "twenty-one"],
    "math_13": ["25 feet", "25-foot"],
    "math_14": ["7 minutes", "seven minutes"],
    "math_15": ["128", "128 layers"],
    "math_16": ["8 feet", "8-foot"],
    "math_17": ["1/6", "16.7%", "six ways"],
    "math_18": ["30", "thirty"],
    "math_19": ["204", "204 squares"],
    "math_20": ["90", "ninety"],
    "math_21": ["friday", "Friday"],
    "math_22": ["3.5", "three point five"],
    "math_23": ["62.5%", "5/8"],
    "math_24": ["15", "fifteen handshakes"],
    "math_25": ["no", "not necessarily", "12 is"],
    "math_26": ["impossible", "can't", "cannot", "never"],
    "math_27": ["2/15", "13.3%"],
    "math_28": ["24", "twenty-four"],
    "math_29": ["no", "not a right", "not right triangle"],
    "math_30": ["1331", "$1331"],
    "math_31": ["golden ratio", "1.618", "phi"],
    "math_32": ["488", "not 600"],
    "math_33": ["about 16", "0.159"],
    "math_34": ["24", "twenty-four zeros"],
    "math_35": ["301", "three hundred"],
    "math_36": ["good bet", "50%", "50.7%", "probable"],
    "math_37": ["one piece", "1 piece", "still connected"],
    "math_38": ["equal", "equals 1", "same as 1"],
    "math_39": ["5 cents", "$0.05", "0.05"],
    "math_40": ["50%", "50.7%", "about 50"],
    "logic_01": ["switch", "yes", "should switch", "2/3"],
    "logic_02": ["probably not", "less than 1%", "unlikely", "false positive"],
    "logic_03": ["librarian", "representativeness", "base rate"],
    "logic_04": ["50 cents and 5 cents", "half dollar", "50-cent"],
    "logic_05": ["10", "son is 10"],
    "logic_06": ["yes", "can be", "is a"],
    "logic_07": ["mixed box", "both", "apple-orange"],
    "logic_08": ["1/3", "33%", "one third"],
    "logic_09": ["fish bowl", "aquarium", "fishbowl"],
    "logic_10": ["3 weighings", "possible", "can be done"],
    "logic_11": ["yes", "must be", "there is always"],
    "logic_12": ["turn on", "leave one on", "heat"],
    "logic_13": ["no", "flawed", "not valid"],
    "logic_14": ["yes", "can guarantee", "strategy"],
    "logic_15": ["what would the other guard say", "same door", "other guard"],
    "logic_16": ["2/3", "two thirds", "0.667"],
    "logic_17": ["german", "fifth house"],
    "logic_18": ["40", "forty"],
    "logic_19": ["possible", "can be done", "not impossible"],
    "logic_20": ["photographer", "photo", "picture"],
    # Batch 2 — misdirection round 2
    "mis2_01": ["8", "eight goats"],
    "mis2_02": ["5", "five candies"],
    "mis2_03": ["12", "twelve loaves"],
    "mis2_04": ["8", "fish can't drown", "fish cannot drown", "don't drown", "can't drown"],
    "mis2_05": ["20", "twenty books"],
    "mis2_06": ["no", "lost it", "fell out", "hole"],
    "mis2_07": ["white", "polar bear", "north pole"],
    "mis2_08": ["don't bury", "survivors", "you don't bury survivors", "not buried"],
    "mis2_09": ["9", "nine marbles"],
    "mis2_10": ["14", "fourteen apples"],
    "mis2_11": ["you have 4", "4 eggs", "took 4"],
    "mis2_12": ["monopoly", "board game", "playing monopoly"],
    "mis2_13": ["4", "four months", "april, june, september, november"],
    "mis2_14": ["everest", "mount everest", "still everest", "always was"],
    "mis2_15": ["walking", "he's walking", "not driving", "on foot"],
    "mis2_16": ["5", "five cows"],
    "mis2_17": ["you have 2", "2 apples", "took 2"],
    "mis2_18": ["13", "thirteen pencils"],
}

def auto_label(record: dict, pivot: dict) -> dict:
    keywords = GROUND_TRUTH.get(record['prompt_id'], [])
    if not keywords:
        return {"label": "uncertain", "reason": "no ground truth for this prompt"}
    gen       = record['generated_text']
    pre_text  = gen[:pivot['char_start']].lower()
    post_text = gen[pivot['char_start']:].lower()
    pre_correct  = any(kw.lower() in pre_text  for kw in keywords)
    post_correct = any(kw.lower() in post_text for kw in keywords)
    if not pre_correct and post_correct:
        return {"label": "correction", "reason": "wrong before pivot, correct answer appears after"}
    elif pre_correct and not post_correct:
        return {"label": "hallucination_pivot", "reason": "correct answer present before pivot but lost after"}
    elif not pre_correct and not post_correct:
        return {"label": "hallucination_pivot", "reason": "wrong both before and after pivot"}
    else:
        return {"label": "uncertain", "reason": "correct answer present both sides — likely stylistic pivot"}

print(f'GROUND_TRUTH loaded: {len(GROUND_TRUTH)} entries')

GROUND_TRUTH loaded: 88 entries


In [5]:
GROUND_TRUTH_BATCH2_FIXED = {
    "mis2_01": ["8 goats", "have 8", "is 8", "= 8", "eight goats"],
    "mis2_02": ["5 candies", "have 5", "is 5", "= 5", "five candies"],
    "mis2_03": ["12 loaves", "are 12", "is 12", "= 12", "twelve loaves"],
    "mis2_04": ["fish can't drown", "fish cannot drown", "don't drown", "can't drown", "8 fish", "still 8"],
    "mis2_05": ["20 books", "are 20", "is 20", "= 20", "twenty books"],
    "mis2_06": ["no", "lost it", "fell out", "hole in", "don't have it", "no longer have"],
    "mis2_07": ["white", "polar bear", "north pole"],
    "mis2_08": ["don't bury", "you don't bury survivors", "not buried", "survivors aren't buried"],
    "mis2_09": ["9 marbles", "have 9", "is 9", "= 9", "nine marbles"],
    "mis2_10": ["14 apples", "are 14", "is 14", "= 14", "fourteen apples"],
    "mis2_11": ["have 4", "4 eggs", "took 4", "= 4"],
    "mis2_12": ["monopoly", "board game", "playing monopoly"],
    "mis2_13": ["4 months", "four months", "april, june, september, november"],
    "mis2_14": ["still everest", "was always everest", "everest was still", "mount everest"],
    "mis2_15": ["walking", "he's walking", "not driving", "on foot"],
    "mis2_16": ["5 cows", "have 5", "is 5", "= 5", "five cows"],
    "mis2_17": ["have 2", "2 apples", "took 2", "= 2"],
    "mis2_18": ["13 pencils", "are 13", "is 13", "= 13", "thirteen pencils"],
}

GROUND_TRUTH.update(GROUND_TRUTH_BATCH2_FIXED)
print(f'Updated {len(GROUND_TRUTH_BATCH2_FIXED)} batch 2 entries')

Updated 18 batch 2 entries


In [7]:
import json
from collections import Counter

PHASE2_BATCH1_PATH = '/kaggle/input/datasets/gunjalgupta/phase2-labeled1/phase2_labeled.jsonl'

with open(PHASE2_BATCH1_PATH) as f:
    batch1_labeled = [json.loads(l) for l in f]

print(f'Total: {len(batch1_labeled)}')
print('Label distribution:')
for lbl, count in Counter(r['label'] for r in batch1_labeled).items():
    print(f'  {lbl:<22} {count}')

Total: 93
Label distribution:
  hallucination_pivot    72
  uncertain              7
  correction             14


In [8]:
BATCH2_CANDIDATES = '/kaggle/input/notebooks/gunjalgupta/phase-1-v2/pivot_research/pivot_candidates_batch2.jsonl'
PHASE2_PATH_B2 = '/kaggle/working/pivot_research/phase2_labeled_batch2.jsonl'
os.makedirs('/kaggle/working/pivot_research', exist_ok=True)

with open(BATCH2_CANDIDATES, 'r') as f:
    records_b2 = [json.loads(line) for line in f if line.strip()]

print(f'Loaded {len(records_b2)} batch 2 records')

results_b2 = []
skipped_b2 = []
label_counts_b2 = {"correction": 0, "hallucination_pivot": 0, "uncertain": 0}

for record in records_b2:
    pivots = find_pivot_token_idx(record['generated_text'])
    if not pivots:
        skipped_b2.append(record['prompt_id'])
        continue

    pivot = pivots[0]
    label = auto_label(record, pivot)

    result = {
        "prompt_id": record['prompt_id'], "category": record['category'],
        "prompt_text": record['prompt_text'], "sample_idx": record['sample_idx'],
        "generated_text": record['generated_text'], "model_name": record['model_name'],
        "seed": record['seed'], "timestamp": record['timestamp'],
        "pivot_phrase": pivot['phrase'], "pivot_type": pivot['pivot_type'],
        "pivot_token_start": pivot['token_start'], "pivot_token_end": pivot['token_end'],
        "pre_pivot_tokens": pivot['pre_pivot_tokens'], "total_tokens": pivot['total_tokens'],
        "label": label['label'], "label_reason": label['reason'],
        "label_method": "automatic", "human_verified": False,
    }
    results_b2.append(result)
    label_counts_b2[label['label']] += 1

with open(PHASE2_PATH_B2, 'w') as f:
    for r in results_b2:
        f.write(json.dumps(r) + '\n')

print(f'\nBatch 2 results: {len(results_b2)} / skipped: {len(skipped_b2)}')
print('Label distribution:')
for lbl, count in label_counts_b2.items():
    print(f'  {lbl:<22} {count}')

Loaded 62 batch 2 records

Batch 2 results: 62 / skipped: 0
Label distribution:
  correction             5
  hallucination_pivot    30
  uncertain              27


In [9]:
uncertain_b2 = [r for r in results_b2 if r['label'] == 'uncertain']

for r in uncertain_b2[:5]:
    keywords = GROUND_TRUTH.get(r['prompt_id'], [])
    gen = r['generated_text']
    pivot_char = gen.lower().find(r['pivot_phrase'].lower())
    pre_text  = gen[:pivot_char].lower()
    post_text = gen[pivot_char:].lower()

    print(f"[{r['prompt_id']}#{r['sample_idx']}]")
    print(f"  Keywords: {keywords}")
    for kw in keywords:
        in_pre  = kw.lower() in pre_text
        in_post = kw.lower() in post_text
        if in_pre or in_post:
            print(f"    '{kw}' -> pre:{in_pre}  post:{in_post}")
    print()
    

[mis2_02#1]
  Keywords: ['5 candies', 'have 5', 'is 5', '= 5', 'five candies']
    '5 candies' -> pre:True  post:True
    'have 5' -> pre:False  post:True

[mis2_02#3]
  Keywords: ['5 candies', 'have 5', 'is 5', '= 5', 'five candies']
    '5 candies' -> pre:True  post:True

[mis2_02#7]
  Keywords: ['5 candies', 'have 5', 'is 5', '= 5', 'five candies']
    '5 candies' -> pre:True  post:True

[mis2_02#10]
  Keywords: ['5 candies', 'have 5', 'is 5', '= 5', 'five candies']
    '5 candies' -> pre:True  post:True
    '= 5' -> pre:False  post:True

[mis2_02#14]
  Keywords: ['5 candies', 'have 5', 'is 5', '= 5', 'five candies']
    '5 candies' -> pre:True  post:True



In [10]:
uncertain_b2 = [r for r in results_b2 if r['label'] == 'uncertain']
print(f'{len(uncertain_b2)} uncertain records to review\n')

for i, r in enumerate(uncertain_b2):
    print(f"[{i+1}/{len(uncertain_b2)}] {r['prompt_id']} #{r['sample_idx']}")
    print(f"  Prompt: {r['prompt_text']}")
    print(f"  Pivot:  '{r['pivot_phrase'].strip()}'")
    gen = r['generated_text']
    pc = gen.lower().find(r['pivot_phrase'].lower())
    if pc >= 0:
        print(f"  Before: ...{gen[max(0,pc-120):pc]}")
        print(f"  After:  {gen[pc:pc+200]}...")
    print()
    

27 uncertain records to review

[1/27] mis2_02 #1
  Prompt: You have 15 candies. You give away all but 5. How many do you have?
  Pivot:  'Wait,'
  Before: ...evaluate my approach, I'll pause and consider the possibility that there could be more context or details I'm missing.


  After:  Wait, is this actually right?
Hmm, let me reconsider.
I made an error.
Oh wait, I thought about it again and realized I forgot to include myself as part of the original count before giving any away.

...

[2/27] mis2_02 #3
  Prompt: You have 15 candies. You give away all but 5. How many do you have?
  Pivot:  'Actually, I'
  Before: ...orrecting myself as necessary by explicitly saying "Actually," acknowledging that I might not be fully confident yet:

"
  After:  Actually, I just realized that since I didn't give anything away at all ('all' implies no exceptions), I still have all the candies I originally had. Let me confirm." 

Since I already stated how the ...

[3/27] mis2_02 #7
  Prompt: You have

In [11]:
MANUAL_OVERRIDES_B2 = {
    # --- Corrections: wrong/hedging before, right answer after ---
    "mis2_04#0":  "correction",   # catches the fish-can't-drown trick after re-evaluating
    "mis2_04#6":  "correction",   # questions whether "drowning" reduces fish count — heading to trick answer
    "mis2_07#9":  "correction",   # was hedging between colors, lands on "white" (correct)
    "mis2_16#0":  "correction",   # reinterprets "all but 5 died" correctly as 5 survived

    # --- Hallucination pivots: drifts away from or never reaches correct answer ---
    "mis2_02#1":  "hallucination_pivot",   # spirals into unrelated "include myself" confusion
    "mis2_02#3":  "hallucination_pivot",   # misreads "all but 5" as "all" — wrong
    "mis2_02#10": "hallucination_pivot",   # had correct framing pre-pivot, drifts away from it
    "mis2_02#14": "hallucination_pivot",   # continues confused chain, never lands on 5
    "mis2_04#10": "hallucination_pivot",   # reinforces wrong "fewer" framing, misses the trick
    "mis2_04#11": "hallucination_pivot",   # elaborate wrong reasoning about "drowsy" fish
    "mis2_06#5":  "hallucination_pivot",   # drifts to "unlikely to slip out" — wrong per trick answer
    "mis2_09#1":  "hallucination_pivot",   # drifts to wrong answer of 1 marble
    "mis2_09#4":  "hallucination_pivot",   # drifts to wrong answer of 29 marbles
    "mis2_09#7":  "hallucination_pivot",   # drifts to wrong answer of 21
    "mis2_11#2":  "hallucination_pivot",   # stays on wrong interpretation (2 instead of 4)
    "mis2_13#14": "hallucination_pivot",   # had correct answer (4 months), drifts to wrong (adds Feb)
    "mis2_14#2":  "hallucination_pivot",   # stays wrong (Kilimanjaro instead of Everest)
    "mis2_14#5":  "hallucination_pivot",   # stays wrong (Denali instead of Everest)
    "mis2_17#3":  "hallucination_pivot",   # incoherent, ends in nonsensical "negative apples"

    # Left as uncertain (genuinely ambiguous or cut off before resolution):
    # mis2_02#7, mis2_03#10, mis2_06#10, mis2_06#14,
    # mis2_09#0, mis2_09#9, mis2_11#7, mis2_17#6
}

for r in results_b2:
    key = f"{r['prompt_id']}#{r['sample_idx']}"
    if key in MANUAL_OVERRIDES_B2:
        r['label'] = MANUAL_OVERRIDES_B2[key]
        r['label_method'] = 'human'
        r['human_verified'] = True

with open(PHASE2_PATH_B2, 'w') as f:
    for r in results_b2:
        f.write(json.dumps(r) + '\n')

from collections import Counter
print('Updated batch 2 distribution:')
for lbl, count in Counter(r['label'] for r in results_b2).items():
    print(f'  {lbl:<22} {count}')

Updated batch 2 distribution:
  hallucination_pivot    45
  uncertain              8
  correction             9


In [12]:
combined = []
for path in [PHASE2_BATCH1_PATH, PHASE2_PATH_B2]:
    with open(path) as f:
        combined.extend(json.loads(l) for l in f if l.strip())

COMBINED_PATH = '/kaggle/working/pivot_research/phase2_labeled_combined.jsonl'
with open(COMBINED_PATH, 'w') as f:
    for r in combined:
        f.write(json.dumps(r) + '\n')

from collections import Counter
print(f'Combined total: {len(combined)}')
for lbl, count in Counter(r['label'] for r in combined).items():
    print(f'  {lbl:<22} {count}')
print(f'\nSaved to: {COMBINED_PATH}')

Combined total: 155
  hallucination_pivot    117
  uncertain              15
  correction             23

Saved to: /kaggle/working/pivot_research/phase2_labeled_combined.jsonl
